In [2]:
import os
# 规避flashinfer采样防止vllm报错
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

In [3]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from vllm import LLM, SamplingParams
import gc

W1028 20:12:01.241000 8649 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1028 20:12:01.243000 8649 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1028 20:12:01.245000 8649 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


In [4]:
print(torch.cuda.is_available())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA version:", torch.version.cuda)

True
PyTorch version: 2.14.1+cu132
CUDA available: True
PyTorch CUDA version: 13.2


In [13]:
PARAMS = {
    'temperature': 0.6,
    'top_p': 0.95,
    'top_k': 20,
    'max_tokens': 512,
}

NameError: name 'temperature' is not defined

#### Load transformers model

In [6]:
model_path = 'models/Qwen3-1.7B'
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [7]:
prompts = [
    "Hello, who are you?",
    "The president of the United States is?",
    "The capital of France is?",
    "The future of AI is?",
]
start_time = time.time()
for p in prompts:
    message = [{
        "role": "user", "content": p
    }]
    text = tokenizer.apply_chat_template(
        message,
        tokenize = False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    input_ids = tokenizer(text, return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **input_ids,
        max_new_tokens=512,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        do_sample=True
    )
    
    output = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
    print("output: ", output)
    print('=' * 60)

print(f"using time {(time.time() - start_time):4f}s")

del model
del output, generated_ids, input_ids, text, message

gc.collect()
torch.cuda.empty_cache()

output:  <|im_start|>user
Hello, who are you?<|im_end|>
<|im_start|>assistant
<think>

</think>

Hello! I'm an AI assistant developed by Alibaba Cloud. I can help with a wide range of tasks, from answering questions to coding, writing, and more. How can I assist you today? 😊<|im_end|>
output:  <|im_start|>user
The president of the United States is?<|im_end|>
<|im_start|>assistant
<think>

</think>

The president of the United States is the head of government and head of state of the United States. The current president is Joe Biden, who was elected in 2020 and has been in office since January 20, 2021. The president is elected by the people of the United States through the Electoral College and serves a term of four years, with the president-elect serving as the vice president until the next election. The president is responsible for leading the executive branch of the government, enforcing federal laws, and representing the United States in foreign affairs.<|im_end|>
output:  <|im_sta

#### Load vLLM model

In [14]:
sampling_params = SamplingParams(temperature=PARAMS['temperature'], top_p=PARAMS['top_p'], max_tokens = PARAMS['max_tokens'])
vllm_model = LLM(
    model = model_path,
    gpu_memory_utilization=0.80,
    max_model_len=8192,
    )

(MainProcess pid=8649) INFO 10-28 20:22:13 [api_utils.py:286] non-default args: {'max_model_len': 8192, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'models/Qwen3-1.7B'}
(MainProcess pid=8649) INFO 10-28 20:22:13 [model.py:733] Resolved architecture: Qwen3ForCausalLM
(MainProcess pid=8649) INFO 10-28 20:22:13 [model.py:2152] Using max model len 8192
(MainProcess pid=8649) INFO 10-28 20:22:13 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


W1028 20:22:23.620000 9265 .venv/lib/python3.14/site-packages/torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1028 20:22:23.621000 9265 .venv/lib/python3.14/site-packages/torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1028 20:22:23.623000 9265 .venv/lib/python3.14/site-packages/torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(EngineCore pid=9265) INFO 10-28 20:23:52 [core.py:133] Initializing a V1 LLM engine (v0.32.0) with config: model='models/Qwen3-1.7B', speculative_config=None, tokenizer='models/Qwen3-1.7B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_m

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:12<00:12, 12.95s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:15<00:00,  6.66s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:15<00:00,  7.60s/it]



(EngineCore pid=9265) INFO 10-28 20:24:27 [default_loader.py:501] Loading weights took 15.23 seconds
(EngineCore pid=9265) INFO 10-28 20:24:29 [model_runner.py:424] Model loading took 3.22 GiB memory and 28.070707 seconds
(EngineCore pid=9265) INFO 10-28 20:24:29 [topk_topp_sampler.py:96] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0.
(EngineCore pid=9265) INFO 10-28 20:24:29 [torch_utils.py:295] Reducing Torch threads from 8 to 1 for serving. Set OMP_NUM_THREADS in the external environment to override.
(EngineCore pid=9265) INFO 10-28 20:24:29 [utils.py:347] Using LBNHC KV cache layout.
(EngineCore pid=9265) INFO 10-28 20:24:43 [caching.py:342] reconstructed serializable fn from standalone compile artifacts. num_artifacts=3 num_submods=29
(EngineCore pid=9265) INFO 10-28 20:24:43 [decorators.py:319] [backbone] Directly load AOT compilation from path /home/msik/.cache/vllm/torch_compile_cache/torch_aot_compile/d0fb9920cecce087a0f8d1ab7e0d0d7d471119e3ef9f673

Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00, 16.16it/s]


(EngineCore pid=9265) INFO 10-28 20:24:50 [model_runner.py:1127] Graph capturing finished in 5 secs, took 0.29 GiB
(EngineCore pid=9265) INFO 10-28 20:24:51 [gpu_worker.py:759] Available KV cache memory: 2.3 GiB
(EngineCore pid=9265) INFO 10-28 20:24:51 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.8000 is equivalent to --gpu-memory-utilization=0.7565 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.8435. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=9265) INFO 10-28 20:24:51 [kv_cache_utils.py:2327] GPU KV cache size: 21,568 tokens, Maximum concurrency for 8,192 tokens per request: 2.63x
(EngineCore pid=9265) INFO 10-28 20:24:51 [kernel_warmup.py:176] JIT kernel warmup starting.
(EngineCore pid=9265) INFO 10-28 20:24:51 [kernel_warmup.py:189] JIT kernel warmup finished in 0.00s.


Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:01<00:00, 26.41it/s]


(EngineCore pid=9265) INFO 10-28 20:24:58 [model_runner.py:1127] Graph capturing finished in 6 secs, took 0.00 GiB
(EngineCore pid=9265) INFO 10-28 20:24:58 [gpu_worker.py:971] CUDA graph pool memory: 0.0 GiB (actual), 0.35 GiB (estimated), difference: 0.35 GiB (37329305600.0%).
(EngineCore pid=9265) INFO 10-28 20:24:58 [gpu_worker.py:1034] Free memory on device (6.81/8.0 GiB) on startup. Desired GPU memory utilization is (0.8, 6.4 GiB). Actual usage is 3.37 GiB for consumed memory (weights + non-torch), 0.72 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=2316946740` (2.16 GiB) to fit into requested memory, or `--kv-cache-memory=2757749248` (2.57 GiB) to fully utilize gpu memory. Current kv cache memory in use is 2.3 GiB.
(EngineCore pid=9265) INFO 10-28 20:25:05 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=9265) INFO 10-28 20:25

In [17]:
messages = [
    [{
        "role": "user", "content": p
    }] for p in prompts
]
start_time = time.time()
outputs = vllm_model.chat(
    messages, 
    sampling_params,
    chat_template_kwargs={"enable_thinking": False},
)

for o in outputs:
    print(o.outputs[0].text)
    print("=" * 60)

print(f"using time {(time.time() - start_time):4f}s")

Rendering conversations:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 4/4 [00:06<00:00,  1.57s/it, est. speed input: 11.77 toks/s, output: 108.98 toks/s]

Hello! I'm an AI assistant developed by Alibaba Cloud. I can help with various tasks, such as answering questions, writing texts, coding, and more. How can I assist you today?
The president of the United States is the head of government and head of state of the United States. The president is elected by the people of the United States through the Electoral College and serves as the commander-in-chief of the United States military. The president is also the chief executive of the federal government and is responsible for enforcing federal laws, conducting foreign policy, and serving as the chief of state. The president's role is outlined in the U.S. Constitution, and the office is currently held by Barack Obama, who served as the 44th president from 2009 to 2017.
The capital of France is Paris.
The future of AI is a topic of great interest and ongoing development, with many experts and researchers predicting various trends and developments. Here's a comprehensive overview of what the fu